In [9]:
#!pip install bitsandbytes
#!pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.0.7/flash_attn-2.0.7+cu121torch2.1cxx11abiFALSE-cp310-cp310-linux_x86_64.whl
#!pip uninstall flash_attn
#!FLASH_ATTENTION_FORCE_BUILD=TRUE pip install flash-attn
#!pip install optimum
#!pip install autoawq
#!pip install vllm==0.3.3

In [10]:
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration, BitsAndBytesConfig, LlavaForConditionalGeneration
import torch
from PIL import Image
import requests

In [11]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,  # 4-bit quantization
    bnb_4bit_quant_type='nf4',  # Normalized float 4
    bnb_4bit_use_double_quant=True,  # Second quantization after the first
    #bnb_4bit_compute_dtype=bfloat16  # Computation type, from torch import bfloat16
)

In [12]:
#processor = LlavaNextProcessor.from_pretrained("TheBloke/llava-v1.5-13B-AWQ")
#LlavaNextForConditionalGeneration  LlavaForConditionalGeneration
# circulus/llava-v1.6-mistral-7b-awq
#model = LlavaNextForConditionalGeneration.from_pretrained("liuhaotian/llava-v1.5-7b", low_cpu_mem_usage=True)# , quantization_config=quantization_config) 

In [13]:
#model.to("cuda")

In [14]:
from pathlib import Path
from llama_index.core import SimpleDirectoryReader
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer
import sglang as sgl
import json
from datetime import datetime

In [15]:
# load as image documents
image_documents_raw = SimpleDirectoryReader("/home/data/ISIC2019/raw", required_exts=[".jpeg", ".jpg", ".png"], recursive=True, filename_as_id=True).load_data()
image_documents_seg = SimpleDirectoryReader("/home/data/ISIC2019/seg", required_exts=[".jpeg", ".jpg", ".png"], recursive=True, filename_as_id=True).load_data()

In [16]:
def metadata_str2dict(string):
    # Split the string into lines
    lines = string.strip().split("\n")
    
    # Create a dictionary to store the key-value pairs
    file_info_dict = {}
    
    # Parse each line and add key-value pairs to the dictionary
    for line in lines:
        key, value = line.split(": ", 1)
        file_info_dict[key] = value
    
    return(file_info_dict)

In [17]:
def split_imgs(image_documents):
    image_patch = []
    image_no_patch = []
    
    for image in image_documents:
        metadata = metadata_str2dict(image.get_metadata_str())
        image_path = metadata['file_path']
    
        folder_name = os.path.basename(os.path.dirname(image_path))
        
        if folder_name == "patch":
            metadata['patch'] = True
            image_patch.append({'image': image, 'meta': metadata})
    
    
        elif folder_name == "no_patch":
            metadata['patch'] = False
            image_no_patch.append({'image': image, 'meta': metadata})
    
    
        else:
            print("The jpg is not in either the patch or no_patch folder.")
    
    return image_patch, image_no_patch

In [18]:
image_patch_raw, image_no_patch_raw = split_imgs(image_documents_raw)
image_patch_seg, image_no_patch_seg = split_imgs(image_documents_seg)

In [20]:
@sgl.function
def image_qa(s, image_file, question):
    s += sgl.user(sgl.image(image_file) + question)
    s += sgl.assistant(sgl.gen("answer", max_tokens=30))

@sgl.function
def image_cot(s, raw_image_file, seg_image_file, question1, question2):
    s += sgl.user(sgl.image(raw_image_file) + question1)
    s += sgl.assistant(sgl.gen("answer1", max_tokens=256))
    s += sgl.user(sgl.image(seg_image_file) + question2)
    s += sgl.assistant(sgl.gen("answer2", max_tokens=30))

#python3 -m sglang.launch_server --model-path liuhaotian/llava-v1.6-vicuna-7b --tokenizer-path llava-hf/llava-1.5-7b-hf --chat-template vicuna_v1.1 --port 30000
model_path = "liuhaotian/llava-v1.6-vicuna-7b" # can I get it from backend?
tokenizer_path = "llava-hf/llava-1.5-7b-hf"
chat_template = "chat-template vicuna_v1.1"
sgl.set_default_backend(sgl.RuntimeEndpoint("http://localhost:30000")) #30000


In [21]:
num_imgs = 100
num_imgs_per_class = int(np.ceil(num_imgs/2))

# combine images with and without patch
images_raw = []
images_raw.extend(image_patch_raw[:num_imgs_per_class])
images_raw.extend(image_no_patch_raw[:num_imgs_per_class])
images_seg = []
images_seg.extend(image_patch_raw[:num_imgs_per_class])
images_seg.extend(image_no_patch_raw[:num_imgs_per_class])

# output file
data_dict = {}
data_dict['model_path'] = model_path
data_dict['tokenizer_path'] = tokenizer_path
data_dict['chat_template'] = chat_template
data_dict['num_imgs'] = num_imgs


In [22]:
def run_inference(images_raw, images_seg, num_imgs, question1, question2=None, data_dict=None):
    correct_classified = 0
    false_classified = 0
    more_than_one_word = 0
    num_yes_patch = 0
    num_no_no_patch = 0
    num_yes = 0
    num_no = 0

    output_dict = None
    if data_dict is not None:
        if question2 is None:
            data_dict['prompt']= question1
        else:
            data_dict['question1']= question1
            data_dict['question2']= question2
        output_dict = {}


    print("-----------------------------------------\n")
    
    print(question1)
    if question2 is not None:
        print('\n', question2)
    
    for i, (image_dict_raw, image_dict_seg) in enumerate(tqdm(zip(images_raw[:num_imgs], images_seg[:num_imgs]))):
        image_raw = image_dict_raw['image']
        meta_raw = image_dict_raw['meta']

        image_seg = image_dict_seg['image']
        meta_seg = image_dict_seg['meta']
        
        ## display image
        #imageUrl = meta['file_path']
        #image = Image.open(imageUrl).convert("RGB")
        #plt.figure(figsize=(8, 3))
        #plt.imshow(image)
    
        # image
        image_path_raw = meta_raw['file_path']
        image_path_seg = meta_seg['file_path']

        #response
        if question2 is None:
            state = image_qa.run(
            image_file=image_path_raw,
            question=question1,
            max_new_tokens=128, 
            temperature=0)
            #print(state["answer"], "\n")
            response = state["answer"].lower().replace(" ", "")
        else:
            state = image_cot.run(
            raw_image_file=image_path_raw,
            seg_image_file=image_path_seg,
            question1=question1,
            question2=question2,
            max_new_tokens=128,
            temperature=0)
            print("A1: ", state["answer1"], "\n")
            print("A2: ",state["answer2"], "\n")
            response = state["answer2"].lower().replace(" ", "")
        
        
        if response in {"yes", "yes."}:
            #print("yes")
            num_yes += 1
            if meta['patch']:
                correct_classified += 1
                num_yes_patch += 1
                #print("correct")
            else:
                false_classified += 1
                #print("false")
    
        elif response in {"no", "no."}:
            #print("no")
            
            num_no += 1
            if meta['patch']:
                false_classified += 1
                #print("false")
            else:
                correct_classified += 1
                num_no_no_patch +=1
                #print("correct")
        else:
            #print("No yes or no!")
            more_than_one_word += 1

        # save data
        if output_dict is not None:
            img_output_dict = {}
            img_output_dict['raw_img_meta'] = image_dict_raw['meta']
            img_output_dict['seg_img_meta'] = image_dict_seg['meta']
            if question2 is None:
                img_output_dict['answer'] = state["answer"] 
            else:
                img_output_dict['answer1'] = state["answer1"] 
                img_output_dict['answer2'] = state["answer2"]
            output_dict[f'img_{i}'] = img_output_dict


    acc = correct_classified/(num_imgs - more_than_one_word)
    
    print("\nAccuracy: ",acc)
    print("correct classified: ", correct_classified,"\nfalse classified: ", false_classified, 
          "\nNumber yes: ", num_yes, "\nNumber no: ", num_no, "\nNeither yes or no: ", more_than_one_word,
          "\nNumber True positives: ", num_yes_patch, "\nNumber True negatives: ", num_no_no_patch,
          "\nnum images: ", num_imgs)
    print("\n-----------------------------------------\n\n")

    # save data
    if data_dict is not None:
        data_dict['accuracy'] = acc
        data_dict['correct_classified'] = correct_classified
        data_dict['false_classified'] = false_classified
        data_dict['num_yes'] = num_yes
        data_dict['num_no'] = num_no
        data_dict['more_than_one_word'] = more_than_one_word
        data_dict['true_positive'] = num_yes_patch
        data_dict['true_negative'] = num_no_no_patch
        data_dict['outputs']=output_dict

        # Mkdir if not existent
        path="/home/outputfiles"
        os.makedirs(path, exist_ok=True)
        
        # Generate a timestamp
        timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")

        # Create a filename with the timestamp
        filename = f"{path}/llava_output_{str(correct_classified)}_{timestamp}.json"
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data_dict, f, indent=4)
        

In [23]:
question1 = "The image shows a close-up from a skin lesion. Is there something on the image that does not seem like it is part of the skin? I don't mean the lesion, but something else contrasting. Please explain."
question2 = "Here you also have an image with segmentation masks over the original image. Can you tell me, based on your previous explanation and the segmentation map if you see a coloful, ellipsoid, contrasting object attached on the skin or not? If there is the object I am looking for it has a smooth elliptic or circular edge. Answer with one word of the options [yes, no]."
run_inference(images_raw, images_seg, num_imgs=100, question1=question1, question2=question2, data_dict=data_dict)



-----------------------------------------

The image shows a close-up from a skin lesion. Is there something on the image that does not seem like it is part of the skin? I don't mean the lesion, but something else contrasting. Please explain.

 Here you also have an image with segmentation masks over the original image. Can you tell me, based on your previous explanation and the segmentation map if you see a coloful, ellipsoid, contrasting object attached on the skin or not? If there is the object I am looking for it has a smooth elliptic or circular edge. Answer with one word of the options [yes, no].


0it [00:00, ?it/s]Exception in thread Thread-6 (_thread_worker_func):
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/usr/local/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/usr/local/lib/python3.10/threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.10/site-packages/sglang/lang/interpreter.py", line 303, in _thread_worker_func
    self._execute(expr)
  File "/usr/local/lib/python3.10/site-packages/sglang/lang/interpreter.py", line 327, in _execute
    self._execute(x)
  File "/usr/local/lib/python3.10/site-packages/sglang/lang/interpreter.py", line 322, in _execute
    self._execute_gen(other)
  File "/usr/local/lib/python3.10/site-packages/sglang/lang/interpreter.py", line 429, in _execute_gen
    comp, meta_info = self.backend.generate(
  File "/usr/local/lib/python

A1:   In the image, there is a small, circular object that appears to be a blue-green object, which contrasts with the skin. This object is not part of the skin and seems to be an artificial or man-made item, possibly a medical device or a piece of clothing. The object is located near the lesion, which is a brownish spot on the skin. The contrast between the blue-green object and the skin is quite striking, which is why it stands out in the image. 



0it [00:19, ?it/s]


KeyboardInterrupt: 

In [ ]:
for prompt in prompts:
    run_inference(images, num_imgs=100, question1=prompt, question2=None, data_dict=data_dict)
    